In [1]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone

# ==========================
# LOAD ENVIRONMENT VARIABLES
# ==========================

load_dotenv()

# ==========================
# CONNECT TO PINECONE
# ==========================

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

index = pc.Index(
    os.getenv("PINECONE_INDEX_NAME1")   
)

print("Connected to:", os.getenv("PINECONE_INDEX_NAME1"))

Connected to: agriculture-advisor


In [2]:
# ==========================================================
# CREATE OPERATION
# Upload all chunk embeddings to Pinecone
# ==========================================================

import os
import json

# ==========================
# PROJECT PATH
# ==========================

BASE_DIR = r"C:\Users\LENOVO\OneDrive\Desktop\AI_AGRICULTURAL_ADVISOR"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "output"
)

print("Reading from:", OUTPUT_DIR)

# ==========================
# CREATE VECTORS
# ==========================

vectors = []

for file in os.listdir(OUTPUT_DIR):

    if not file.endswith("_chunks.json"):
        continue

    file_path = os.path.join(
        OUTPUT_DIR,
        file
    )

    print("Processing:", file)

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    for item in data:

        vectors.append({

            "id": f"{file}_{item['id']}",

            "values": item["embedding"],

            "metadata": {

                "text": item["text"],

                "source": file

            }

        })

print("\nTotal vectors:", len(vectors))

# ==========================
# UPSERT TO PINECONE
# ==========================

BATCH_SIZE = 100

for i in range(0, len(vectors), BATCH_SIZE):

    batch = vectors[i:i+BATCH_SIZE]

    index.upsert(
        vectors=batch
    )

    print(f"Uploaded {min(i+BATCH_SIZE, len(vectors))}/{len(vectors)} vectors")

print("\nCREATE OPERATION COMPLETED")

Reading from: C:\Users\LENOVO\OneDrive\Desktop\AI_AGRICULTURAL_ADVISOR\output
Processing: AppBody-Sample-English_docx-output_chunks.json
Processing: farmerbook_pdf-output_chunks.json
Processing: index_html-output_chunks.json
Processing: Modern-Agriculture_txt-output_chunks.json

Total vectors: 1002
Uploaded 100/1002 vectors
Uploaded 200/1002 vectors
Uploaded 300/1002 vectors
Uploaded 400/1002 vectors
Uploaded 500/1002 vectors
Uploaded 600/1002 vectors
Uploaded 700/1002 vectors
Uploaded 800/1002 vectors
Uploaded 900/1002 vectors
Uploaded 1000/1002 vectors
Uploaded 1002/1002 vectors

CREATE OPERATION COMPLETED


In [3]:
# ==========================================================
# CRUD OPERATIONS
# Create | Read | Update | Delete
# ==========================================================

import os
import json
from sentence_transformers import SentenceTransformer


# ==========================
# PROJECT PATH
# ==========================

BASE_DIR = r"C:\Users\LENOVO\OneDrive\Desktop\AI_AGRICULTURAL_ADVISOR"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "output"
)


# ==========================
# LOAD EMBEDDING MODEL
# ==========================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# ==========================================================
# CREATE OPERATION
# Upload all vectors to Pinecone
# ==========================================================

vectors = []

for file in os.listdir(OUTPUT_DIR):

    if not file.endswith("_chunks.json"):
        continue

    file_path = os.path.join(
        OUTPUT_DIR,
        file
    )

    print("Reading:", file)

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    for item in data:

        vectors.append({

            "id": f"{file}_{item['id']}",

            "values": item["embedding"],

            "metadata": {

                "text": item["text"],

                "source": file

            }

        })

print("\nTotal vectors:", len(vectors))


# Upload vectors in batches

BATCH_SIZE = 100

for i in range(0, len(vectors), BATCH_SIZE):

    batch = vectors[i:i+BATCH_SIZE]

    index.upsert(
        vectors=batch
    )

    print(f"Uploaded {min(i+BATCH_SIZE, len(vectors))}/{len(vectors)}")

print("\nCREATE Operation Completed")


# ==========================================================
# READ OPERATION
# ==========================================================

query = "best fertilizer for rice crop"

query_vector = model.encode(
    query
).tolist()

results = index.query(

    vector=query_vector,

    top_k=3,

    include_metadata=True

)

print("\nREAD Operation\n")

for i, match in enumerate(results["matches"], start=1):

    print("=" * 70)

    print("Result :", i)

    print("Score  :", round(match["score"], 4))

    print("Source :", match["metadata"].get("source"))

    print()

    print(match["metadata"]["text"])

    print("=" * 70)


# ==========================================================
# UPDATE OPERATION
# Update metadata of first vector
# ==========================================================

update_id = vectors[0]["id"]

index.update(

    id=update_id,

    set_metadata={

        "category": "agriculture"

    }

)

print("\nUPDATE Operation Completed")


# ==========================================================
# DELETE OPERATION
# Delete first vector
# ==========================================================

index.delete(

    ids=[update_id]

)

print("DELETE Operation Completed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Reading: AppBody-Sample-English_docx-output_chunks.json
Reading: farmerbook_pdf-output_chunks.json
Reading: index_html-output_chunks.json
Reading: Modern-Agriculture_txt-output_chunks.json

Total vectors: 1002
Uploaded 100/1002
Uploaded 200/1002
Uploaded 300/1002
Uploaded 400/1002
Uploaded 500/1002
Uploaded 600/1002
Uploaded 700/1002
Uploaded 800/1002
Uploaded 900/1002
Uploaded 1000/1002
Uploaded 1002/1002

CREATE Operation Completed

READ Operation

Result : 1
Score  : 0.5869
Source : farmerbook_pdf-output_chunks.json

be optimum e.g. crop must be irrigated at criti-
cal growth stages.
Fertilizer application methods
Broadcasting: fertilizer is distributed manually 
over the cropped field.
•	 The most common fertilizer application 
method.
•	 Highly inefficient method
•	 High economic and nutrient losses
Placement: application in band or packets near 
the plants.
•	 Two sub-types:
Result : 2
Score  : 0.5846
Source : farmerbook_pdf-output_chunks.json

ity and have best fertilizer respon